# 03. Hybrid and Reranking (`TextExperimentAdapter` + `ExperimentTracker`)

In [1]:
from pathlib import Path

import mlflow
import pandas as pd

from meme_organizer.config import MLFLOW_EXPERIMENT_FUSION, MLFLOW_EXPERIMENT_TEXT
from meme_organizer.evaluation.metrics.experiment_tracker import ExperimentTracker
from meme_organizer.evaluation.pipeline.evaluation_dataset_preparer import EvaluationDatasetPreparer
from meme_organizer.retrieval.fusion.hybrid_retriever import HybridRetriever
from meme_organizer.retrieval.text.bm25_retriever import BM25Retriever
from meme_organizer.retrieval.text.dense_text_retriever import DenseTextRetriever
from meme_organizer.retrieval.text.fine_tuned_dense_retriever import FineTunedDenseRetriever
from meme_organizer.retrieval.text.text_experiment_adapter import TextExperimentAdapter

ROOT = Path("..").resolve()
split_data = EvaluationDatasetPreparer().prepare(
    ROOT / "data" / "gold" / "gold_dataset.jsonl",
    ROOT / "data" / "splits" / "splits_v1.json",
    "test",
)
print(f"Corpus: {len(split_data.corpus_docs)} | Test queries: {len(split_data.queries)}")

2026/11/27 16:55:29 INFO mlflow.agent.hint: Load the `mlflow-trace-instrumentation` skill at /home/alex/Projects/meme-organizer/.venv/lib/python3.13/site-packages/mlflow/assistant/skills/mlflow-trace-instrumentation/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


Corpus: 318 | Test queries: 153


In [2]:
with ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_FUSION) as exp:
    res_rrf = exp.create_pipeline(
        TextExperimentAdapter("hybrid-rrf", HybridRetriever(mode="rrf", model_name="hybrid-rrf"), split_data)
    ).run()
print("Hybrid RRF:", res_rrf["quality"])

2026/11/27 16:55:32 INFO mlflow.tracking.fluent: Experiment with name 'meme-organizer-fusion' does not exist. Creating a new experiment.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🏃 View run hybrid-rrf at: http://localhost:5000/#/experiments/13/runs/e84433caf24e4666977e3426efa42eb1
🧪 View experiment at: http://localhost:5000/#/experiments/13
Hybrid RRF: {'recall_at_1': 0.6078, 'recall_at_5': 0.7059, 'mrr_at_5': 0.6444, 'ndcg_at_5': 0.6598, 'total_queries': 153}


In [3]:
with ExperimentTracker(experiment_name=MLFLOW_EXPERIMENT_FUSION) as exp:
    res_optuna = exp.create_pipeline(
        TextExperimentAdapter("hybrid-optuna", HybridRetriever(mode="optuna", model_name="hybrid-optuna"), split_data)
    ).run()
print("Hybrid Optuna:", res_optuna["quality"])

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

🏃 View run hybrid-optuna at: http://localhost:5000/#/experiments/13/runs/5cebfb5d7e41491888289b20d357b5c6
🧪 View experiment at: http://localhost:5000/#/experiments/13
Hybrid Optuna: {'recall_at_1': 0.6536, 'recall_at_5': 0.7582, 'mrr_at_5': 0.6983, 'ndcg_at_5': 0.7135, 'total_queries': 153}
